In [ ]:
import os
import sys
import json
from pathlib import Path
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

# Ensure `week1/scraper.py` is importable from this nested notebook folder.
for p in [Path.cwd(), *Path.cwd().parents]:
    scraper_dir = p / "week1"
    if (scraper_dir / "scraper.py").exists():
        sys.path.insert(0, str(scraper_dir))
        break

from scraper import fetch_website_contents, fetch_website_links


In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if google_api_key:
    print("API Key found and running")
else:
    print("API key not found")


In [ ]:
links=fetch_website_links("https://beebuddy.in/")
links

In [ ]:
link_system_prompt = """
You are given a list of links from one website.
Read and Select links that are most relevant as discovery-ready class/course detail pages.

Return ONLY a JSON object in this exact shape:
{
  "links": [
    "https://example.com/class/some-class-page"
  ]
}

Selection rules:
- Prefer detail pages that containg class name, location/address, date/timing (for this site, URLs containing /class/ are usually relevant).
- Exclude homepage, contact, blog, login, policy.
- Keep absolute URLs only.
- Return up to 40 best links.
"""


In [ ]:
def get_links_user_prompt(url):
    from urllib.parse import urljoin

    user_prompt = f"""
              Review these links and read the name of the class, timing, location. 
              Fetch the google reviews of the class on the web.
              Page URL: {url}
              Return only valid JSON matching the system schema.
                  """
    raw_links = fetch_website_links(url)

    # Convert to absolute URLs, de-duplicate, and cap size.
    seen = set()
    links = []
    for link in raw_links:
        absolute_link = urljoin(url, link)
        if absolute_link not in seen:
            seen.add(absolute_link)
            links.append(absolute_link)
        if len(links) >= 300:
            break

    user_prompt += "\nCandidate links:\n" + "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://beebuddy.in/"))

In [ ]:
def _parse_json_object(content):
    import json
    from json import JSONDecoder

    # Handle plain JSON first.
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        pass

    # Decode first valid JSON value from noisy output.
    decoder = JSONDecoder()
    for i, ch in enumerate(content):
        if ch not in "[{":
            continue
        try:
            obj, _ = decoder.raw_decode(content[i:])
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError:
            continue

    raise ValueError("Model response did not contain a valid JSON object.")


def select_relevant_links(url):
    import re
    from urllib.parse import urljoin

    candidate_prompt = get_links_user_prompt(url)
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": candidate_prompt}
        ],
        response_format={"type": "json_object"},
        temperature=0
    )
    result = response.choices[0].message.content or ""
    parsed = _parse_json_object(result)

    model_links = parsed.get("links", []) if isinstance(parsed, dict) else []
    model_links = [urljoin(url, x) for x in model_links if isinstance(x, str)]

    # Heuristic fallback when model returns empty or low-signal output.
    if not model_links:
        all_urls = re.findall(r"https?://[^\s\"'<>]+", candidate_prompt)
        model_links = [u for u in all_urls if "/class/" in u.lower()]

    # De-duplicate and cap.
    deduped = []
    seen = set()
    for u in model_links:
        if u not in seen:
            seen.add(u)
            deduped.append(u)
        if len(deduped) >= 40:
            break

    links = {"links": deduped}
    print(f"Found {len(links['links'])} relevant links")
    return links


MODEL = "gemini-2.5-flash-lite"
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
select_relevant_links("https://beebuddy.in/")